## 1. Зависимости

In [3]:
!pip install mlflow evidently scikit-learn pandas numpy matplotlib seaborn -qqq

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from evidently import ColumnMapping
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.metrics import DatasetDriftMetric, DataDriftTable

import matplotlib.pyplot as plt
import seaborn as sns


## 2. MLflow

In [ ]:

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")


mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")


EXPERIMENT_NAME = "iris_classification"
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

## 3. Load and Prepare Data

In [ ]:
iris = load_iris()
feature_names = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
target_names = list(iris.target_names)

df = pd.DataFrame(iris.data, columns=feature_names)
df['target'] = iris.target
df['species'] = df['target'].map(lambda x: target_names[x])

print(f"Dataset shape: {df.shape}")
print(f"\nFeatures: {feature_names}")
print(f"Classes: {target_names}")
df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, feature in enumerate(feature_names):
    ax = axes[idx // 2, idx % 2]
    for species in target_names:
        data = df[df['species'] == species][feature]
        ax.hist(data, alpha=0.5, label=species, bins=15)
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.set_title(f'Distribution of {feature}')

plt.tight_layout()
plt.savefig('/tmp/feature_distributions.png', dpi=150)
plt.show()

In [ ]:

X = df[feature_names]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 4. MLflow  трекинг



In [ ]:
def train_and_log_model(model, model_name, params):

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)

        model.fit(X_train, y_train)

        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        train_accuracy = accuracy_score(y_train, y_pred_train)
        test_accuracy = accuracy_score(y_test, y_pred_test)
        
        # Cross-validation
        cv_scores = cross_val_score(model, X, y, cv=5)
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()

        mlflow.log_metrics({
            "train_accuracy": train_accuracy,
            "test_accuracy": test_accuracy,
            "cv_mean_accuracy": cv_mean,
            "cv_std_accuracy": cv_std
        })
        
        # Log model
        mlflow.sklearn.log_model(
            model, 
            "model",
            registered_model_name=f"iris_{model_name.lower().replace(' ', '_')}"
        )
        
        # Log confusion matrix as artifact
        cm = confusion_matrix(y_test, y_pred_test)
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=target_names, yticklabels=target_names, ax=ax)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        ax.set_title(f'Confusion Matrix - {model_name}')
        plt.tight_layout()
        
        cm_path = f'/tmp/confusion_matrix_{model_name.replace(" ", "_")}.png'
        plt.savefig(cm_path)
        mlflow.log_artifact(cm_path)
        plt.close()
        
        print(f"\n{model_name}:")
        print(f"  Train Accuracy: {train_accuracy:.4f}")
        print(f"  Test Accuracy: {test_accuracy:.4f}")
        print(f"  CV Accuracy: {cv_mean:.4f} (+/- {cv_std:.4f})")
        
        return {
            "model_name": model_name,
            "train_accuracy": train_accuracy,
            "test_accuracy": test_accuracy,
            "cv_mean": cv_mean
        }

In [ ]:
# Train multiple models
results = []

# 1. Random Forest
rf_params = {"n_estimators": 100, "max_depth": 5, "random_state": 42}
rf_model = RandomForestClassifier(**rf_params)
results.append(train_and_log_model(rf_model, "Random Forest", rf_params))

# 2. Gradient Boosting
gb_params = {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1, "random_state": 42}
gb_model = GradientBoostingClassifier(**gb_params)
results.append(train_and_log_model(gb_model, "Gradient Boosting", gb_params))

# 3. Logistic Regression
lr_params = {"C": 1.0, "max_iter": 200, "random_state": 42}
lr_model = LogisticRegression(**lr_params)
results.append(train_and_log_model(lr_model, "Logistic Regression", lr_params))

# Summary
results_df = pd.DataFrame(results)
print("\n" + "="*50)
print("EXPERIMENT SUMMARY")
print("="*50)
print(results_df.to_string(index=False))

In [ ]:
# Select best model and register it for production
best_model_name = results_df.loc[results_df['test_accuracy'].idxmax(), 'model_name']
print(f"Best model: {best_model_name}")

# Transition to Production stage
client = MlflowClient()
model_registry_name = f"iris_{best_model_name.lower().replace(' ', '_')}"

try:
    latest_version = client.get_latest_versions(model_registry_name, stages=["None"])[0]
    client.transition_model_version_stage(
        name=model_registry_name,
        version=latest_version.version,
        stage="Production"
    )
    print(f"Model {model_registry_name} version {latest_version.version} transitioned to Production")
except Exception as e:
    print(f"Note: Could not transition model stage: {e}")

## 5. EvidentlyAI


In [ ]:
reference_data = X_train.copy()
reference_data['target'] = y_train.values

current_data = X_test.copy()
current_data['target'] = y_test.values

print(f"{reference_data.shape}")
print(f" {current_data.shape}")

In [ ]:
column_mapping = ColumnMapping(
    target='target',
    numerical_features=feature_names
)

drift_report = Report(metrics=[
    DatasetDriftMetric(),
    DataDriftTable()
])

drift_report.run(
    reference_data=reference_data,
    current_data=current_data,
    column_mapping=column_mapping
)

drift_report

In [ ]:
drift_report.save_html('/tmp/drift_report.html')

drift_results = drift_report.as_dict()
dataset_drift = drift_results['metrics'][0]['result']


## 6. Отчет качества данных

In [ ]:
quality_report = Report(metrics=[
    DataQualityPreset()
])

quality_report.run(
    reference_data=reference_data,
    current_data=current_data,
    column_mapping=column_mapping
)

quality_report.save_html('/tmp/data_quality_report.html')

quality_report

In [ ]:
print("\n" + "="*60)
print("PIPELINE SUMMARY")
print("="*60)
print(f"\nMLflow Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment Name: {EXPERIMENT_NAME}")
print(f"Best Model: {best_model_name}")
print(f"\nGenerated Reports:")
print(f"  - /tmp/feature_distributions.png")
print(f"  - /tmp/drift_report.html")
print(f"  - /tmp/drift_report_detected.html")
print(f"  - /tmp/data_quality_report.html")
print("\nPipeline completed successfully!")